# Norm-supervised merge (SCPS) experiments on Colab

This notebook reproduces the experiment matrix from `scripts/run_experiments_merge.sh`. It clones a selected Git ref, creates an isolated virtual environment, runs each configuration through `scripts/test_merge.py`, and writes CSV results directly to Google Drive.

Point `DRIVE_MODEL_OVERRIDES` at a trained `merge_ME_basic` checkpoint from `RQL-Comparison` (e.g. `train_merge_colab.ipynb`). Use a distinct `RUN_NAME` when running multiple Colab sessions in parallel.

In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

In [ ]:
from datetime import datetime, timezone

REPO_URL = "https://github.com/thowell332/state-wise-constrained-policy-shaping.git"  #@param {type:"string"}
REPO_REF = "thowell332/results-galore"  #@param {type:"string"}

NUM_EPISODES = 1000  #@param {type:"integer"}
BASE_SEED = 42  #@param {type:"integer"}
ENVIRONMENTS = ["MERGE_BASIC"]  # Valid: MERGE_BASIC
FORCE_WRITE = True  #@param {type:"boolean"}
FILE_PREFIX = "MERGE"  # Preserves the naming used by run_experiments_merge.sh

# Copy real model zips from Drive into the cloned repo.
# Keys are paths relative to the project root.
DRIVE_MODEL_OVERRIDES = {
    "models/merge_ME_basic.zip": "/content/drive/MyDrive/aaai2027/RQL-Comparison/models/merge_basic_seed1/model.zip",
}

# Each tuple is: (profile, method, value, enforce_filter).
# value must be None except for adaptive/fixed methods.
EXPERIMENTS = [
    ("merge_courtesy", "nop", None, False),
    ("merge_courtesy", "nop", None, True),
    ("merge_courtesy", "naive", None, False),
    ("merge_courtesy", "naive", None, True),
    ("merge_courtesy", "adaptive", "0.05", False),
    ("merge_courtesy", "adaptive", "0.05", True),
    ("merge_courtesy", "fixed", "1.00", False),
    ("merge_courtesy", "fixed", "1.00", True),
    ("merge_courtesy", "projection", None, False),
    ("merge_courtesy", "projection", None, True),
    # ("merge_courtesy", "adaptive", "0.0316", True),
    # ("merge_courtesy", "adaptive", "0.3162", True),
    # ("merge_courtesy", "adaptive", "3.1623", True),
    # ("merge_courtesy", "adaptive", "10.000", True),
]

RUN_NAME = datetime.now(timezone.utc).strftime("merge_run_%Y%m%dT%H%M%SZ")  #@param {type:"string"}
DRIVE_RESULTS_ROOT = "/content/drive/MyDrive/aaai2027/state-wise-constrained-policy-shaping"  #@param {type:"string"}

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/state-wise-constrained-policy-shaping")
VENV_DIR = Path("/content/venvs/state-wise-constrained-policy-shaping")
ENV_MODEL_FILES = {
    "MERGE_BASIC": "models/merge_ME_basic.zip",
}

os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["OFFSCREEN_RENDERING"] = "1"

for path in (PROJECT_DIR, VENV_DIR):
    if path.exists():
        shutil.rmtree(path)

subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", REPO_REF], check=True)

for relative_model, drive_src in DRIVE_MODEL_OVERRIDES.items():
    dst = PROJECT_DIR / relative_model
    src = Path(drive_src).expanduser()
    if not src.is_file() or src.stat().st_size == 0:
        raise FileNotFoundError(
            f"Drive model override not found or empty: {src}\n"
            f"Copy the real zip to Drive and update DRIVE_MODEL_OVERRIDES."
        )
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    shutil.copy2(src, dst)
    print(f"Installed model override: {dst} <- {src} ({dst.stat().st_size:,} bytes)")


def create_venv(venv_dir: Path) -> Path:
    venv_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["apt-get", "install", "-y", "python3-venv", f"python{sys.version_info.major}.{sys.version_info.minor}-venv"],
        check=False,
    )
    result = subprocess.run(
        [sys.executable, "-m", "venv", "--system-site-packages", str(venv_dir)],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print("stdlib venv failed; falling back to virtualenv")
        print(result.stderr)
        if venv_dir.exists():
            shutil.rmtree(venv_dir)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "virtualenv"], check=True)
        subprocess.run(
            [sys.executable, "-m", "virtualenv", "--system-site-packages", str(venv_dir)],
            check=True,
        )
    return venv_dir / "bin" / "python"


VENV_PYTHON = create_venv(VENV_DIR)

os.environ["PYTHONPATH"] = str(PROJECT_DIR) + (
    os.pathsep + os.environ["PYTHONPATH"] if os.environ.get("PYTHONPATH") else ""
)

subprocess.run([str(VENV_PYTHON), "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run(
    [str(VENV_PYTHON), "-m", "pip", "uninstall", "-y", "stable-baselines3", "highway-env"],
    check=False,
)
subprocess.run(
    [
        str(VENV_PYTHON), "-m", "pip", "install",
        "gymnasium>=1.0.0", "numpy", "pandas", "scipy", "matplotlib", "tensorboard",
    ],
    check=True,
)
subprocess.run(
    [str(VENV_PYTHON), "-m", "pip", "install", "--no-deps", "-e", str(PROJECT_DIR)],
    check=True,
)

for required in (
    PROJECT_DIR / "scripts/test_merge.py",
    PROJECT_DIR / "stable_baselines3" / "dqn_ME",
    PROJECT_DIR / "highway_env",
    PROJECT_DIR / "supervisor",
):
    if not required.exists():
        raise FileNotFoundError(
            f"Selected Git ref is missing required path: {required}\n"
            "Push the vendored stable_baselines3/highway_env packages to the branch you clone."
        )

probe = subprocess.run(
    [
        str(VENV_PYTHON), "-c",
        "from supervisor import DiscreteSupervisor; "
        "assert 'merge_courtesy' in DiscreteSupervisor.PROFILES; "
        "print('supervisor ok:', sorted(DiscreteSupervisor.PROFILES))",
    ],
    check=True,
    capture_output=True,
    text=True,
)
print(probe.stdout)

missing_models = []
for env_name in ENVIRONMENTS:
    if env_name not in ENV_MODEL_FILES:
        raise ValueError(f"Unknown environment {env_name!r}. Valid: {sorted(ENV_MODEL_FILES)}")
    model_path = PROJECT_DIR / ENV_MODEL_FILES[env_name]
    if model_path.is_symlink() and not model_path.exists():
        missing_models.append(f"{model_path} (broken symlink)")
    elif not model_path.is_file() or model_path.stat().st_size == 0:
        missing_models.append(str(model_path))
if missing_models:
    raise FileNotFoundError(
        "Missing model files after clone/overrides. Add them to DRIVE_MODEL_OVERRIDES:\n  - "
        + "\n  - ".join(missing_models)
    )

print(f"Project: {PROJECT_DIR}")
print(f"Python:  {VENV_PYTHON}")
print(f"PYTHONPATH includes: {PROJECT_DIR}")

In [ ]:
import json
import os
from pathlib import Path

RUN_DIR = Path(DRIVE_RESULTS_ROOT).expanduser() / RUN_NAME
RESULTS_DIR = RUN_DIR / "results"
RUN_DIR.mkdir(parents=True, exist_ok=True)

manifest = {
    "repo_url": REPO_URL,
    "repo_ref": REPO_REF,
    "num_episodes": NUM_EPISODES,
    "base_seed": BASE_SEED,
    "environments": ENVIRONMENTS,
    "force_write": FORCE_WRITE,
    "file_prefix": FILE_PREFIX,
    "drive_model_overrides": DRIVE_MODEL_OVERRIDES,
    "experiments": EXPERIMENTS,
}
(RUN_DIR / "run_parameters.json").write_text(json.dumps(manifest, indent=2))


def run_and_tee(command, log_handle):
    print("$", " ".join(command))
    process = subprocess.Popen(
        command,
        cwd=PROJECT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONPATH": str(PROJECT_DIR)},
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        log_handle.write(line)
        log_handle.flush()
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)


with (RUN_DIR / "run.log").open("a", buffering=1) as log:
    for env_name in ENVIRONMENTS:
        for profile, method, value, filter_enabled in EXPERIMENTS:
            sub_dir = f"{method}_filtered" if filter_enabled else f"{method}_unfiltered"
            out_dir = RESULTS_DIR / env_name / profile / sub_dir
            out_dir.mkdir(parents=True, exist_ok=True)
            if method in ("adaptive", "fixed"):
                out_path = out_dir / f"{FILE_PREFIX}_{env_name}_{value}.csv"
            else:
                out_path = out_dir / f"{FILE_PREFIX}_{env_name}.csv"

            if out_path.is_file() and not FORCE_WRITE:
                print(f"Skipping existing results: {out_path}")
                continue
            if out_path.is_file() and FORCE_WRITE:
                print(f"Overwriting existing results: {out_path}")

            cmd = [
                str(VENV_PYTHON),
                str(PROJECT_DIR / "scripts" / "test_merge.py"),
                "--profile", profile,
                "--method", method,
                "--episodes", str(NUM_EPISODES),
                "--seed", str(BASE_SEED),
                "--env", env_name,
                "--output", str(out_path),
            ]
            if value is not None:
                cmd.extend(["--value", str(value)])
            if filter_enabled:
                cmd.append("--filter")

            run_and_tee(cmd, log)

print(f"Results saved to: {RUN_DIR}")